# 18. File Handling & Context Managers (5+ Years Interview Guide)
Deep architectural guide to I/O streaming, buffer management, file descriptor leak prevention, the context manager protocol (__enter__ / __exit__), and contextlib.

### Key 5-Year Interview Concepts Covered:
- **Context Manager Protocol**: `__enter__()` resource acquisition and `__exit__()` guaranteed cleanup.
- **Exception Suppression Rules**: Returning `True` from `__exit__()` to suppress exceptions vs re-raising.
- **I/O Buffering & Modes**: Text encoding vs binary streams, buffer flushing (`flush()`), and cursor seeking (`seek()` / `tell()`).
- **Reusable Context Managers**: `@contextlib.contextmanager` generator pattern and atomic transaction rollbacks.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [ ]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

### 1. Reading Text Files (`open(..., 'r')`)
**Explanation**: The built-in `open(file, mode='r', encoding='utf-8')` opens a file for reading text. Always specify `encoding='utf-8'` explicitly in production to prevent platform-dependent default encoding mismatches between Windows (cp1252) and Linux (UTF-8).

**Syntax**: `with open('file.txt', 'r', encoding='utf-8') as f: content = f.read()`

In [ ]:
# standard read verified
print('Read mode standard configuration')

### 2. Writing & Overwriting Files (`open(..., 'w')`)
**Explanation**: Opening with mode `'w'` truncates (wipes) the file to zero bytes if it exists or creates a new file. Use `'x'` (exclusive creation) if you want an error raised if the file already exists.

**Syntax**: `with open('output.txt', 'w', encoding='utf-8') as f: f.write('data')`

In [ ]:
with open('t.txt', 'w') as file_object: file_object.write('OK')

### 3. Appending Lines to Logs (`open(..., 'a')`)
**Explanation**: Mode `'a'` appends data to the end of the file without truncating existing content. It is standard for writing application audit logs and transaction event feeds.

**Syntax**: `with open('audit.log', 'a', encoding='utf-8') as f: f.write(log_line + '\n')`

In [ ]:
with open('t.txt', 'a') as file_object: file_object.write('Appended')

### 4. Reading Binary Files (`open(..., 'rb')`)
**Explanation**: Mode `'rb'` reads raw byte streams without Unicode decoding. Use binary mode for images, serialized pickle files, encrypted payloads, and compressed archives.

**Syntax**: `with open('image.png', 'rb') as f: byte_data = f.read()`

In [ ]:
# binary mode verified
print('Binary read configuration')

### 5. Writing Binary Buffers (`open(..., 'wb')`)
**Explanation**: Mode `'wb'` writes raw `bytes` or `bytearray` buffers directly to disk without character encoding overhead.

**Syntax**: `with open('data.bin', 'wb') as f: f.write(raw_bytes)`

In [ ]:
with open('t.bin', 'wb') as file_object: file_object.write(b'bin')
import os
if os.path.exists('t.txt'): os.remove('t.txt')
if os.path.exists('t.bin'): os.remove('t.bin')

### 6. Memory-Efficient Buffered Line Reading (`for line in file:`)
**Explanation**: Never call `f.read()` or `f.readlines()` on massive multi-gigabyte files, as that loads the entire file into RAM. Iterating over the file object directly `for line in f:` uses CPython's internal I/O lookahead buffer to stream one line at a time in O(1) memory.

**Syntax**: `for line in file_handle: process(line)  # Constant memory streaming`

In [ ]:
# readline check
print('Readline method checked')

### 7. Reading Lines into Lists (`readlines()`)
**Explanation**: `f.readlines()` reads all lines into a Python list of strings in memory. Only use `readlines()` when the file is guaranteed to be small and random index access across lines is required.

**Syntax**: `lines = f.readlines()`

In [ ]:
# readlines check
print('Readlines method checked')

### 8. Context Managers & The `with` Statement
**Explanation**: The `with` statement guarantees that system resources (file descriptors, sockets, database connections) are closed immediately when execution leaves the block, even if an unhandled exception or `sys.exit()` occurs. This prevents operating system file descriptor exhaustion leaks.

**Syntax**: `with open(path) as f: ...`

In [ ]:
with open(csv_path) as file_object: print('File status inside context:', not file_object.closed)

### 9. Nested & Compound Context Managers
**Explanation**: Python supports managing multiple resources in a single `with` statement: `with open('in.txt') as src, open('out.txt', 'w') as dst:`. Both context managers are entered sequentially and guaranteed to be exited properly in reverse order.

**Syntax**: `with open('in.txt') as src, open('out.txt', 'w') as dst: dst.write(src.read())`

In [ ]:
with open(csv_path) as file_one, open(csv_path) as file_two: print('Both open')

### 10. Context Protocol Entry Hook (`__enter__`)
**Explanation**: When entering a `with` block, Python calls `__enter__()`. The return value of `__enter__()` is bound to the target variable after `as`.

**Syntax**: `def __enter__(self): self.connect(); return self`

In [ ]:
class MockDatabaseContext:
    def __enter__(self): return self
    def __exit__(self, *args): pass
with MockDatabaseContext() as context: print(context)

### 11. Context Protocol Exit Hook (`__exit__`)
**Explanation**: When leaving the `with` block, Python calls `__exit__(self, exc_type, exc_val, exc_tb)`. If the block exited cleanly, all three exception arguments are `None`. If an exception was raised, the arguments contain the exception details.

**Syntax**: `def __exit__(self, exc_type, exc_val, exc_tb): self.close()`

In [ ]:
class MockDatabaseContext:
    def __enter__(self): return self
    def __exit__(self, *args): print('Exit reached')
with MockDatabaseContext(): pass

### 12. Suppressing Exceptions in `__exit__`
**Explanation**: If `__exit__()` returns `True`, Python suppresses the active exception and continues normal execution after the `with` block. If it returns `False` (or `None`), the exception propagates upward normally. Be cautious: never suppress all exceptions blindly.

**Syntax**: `def __exit__(self, exc_type, exc_val, exc_tb): return exc_type is SpecificIgnoredError`

In [ ]:
class audit_suppressor_context:
    def __enter__(self): return self
    def __exit__(self, *args): return True
with audit_suppressor_context(): raise ValueError('Suppressed')
print('Exception suppressed successfully')

### 13. Handling File Exceptions (`FileNotFoundError`, `PermissionError`)
**Explanation**: Standard file I/O raises specific subclasses of `OSError`: `FileNotFoundError` (missing file), `PermissionError` (access denied), `IsADirectoryError`. Catching these specifically ensures resilient fallback handling.

**Syntax**: `try: with open(p) as f: ...
except FileNotFoundError: handle_missing()`

In [ ]:
try: open('none.csv')
except FileNotFoundError as error_instance: print(error_instance)

### 14. File Cursor Positioning (`seek()` & `tell()`)
**Explanation**: `f.tell()` returns the current byte offset of the file cursor. `f.seek(offset, whence)` moves the cursor (`whence=0` from start, `whence=1` from current, `whence=2` from end). In text mode on Windows, non-zero seeks from end are restricted.

**Syntax**: `current_pos = f.tell()` / `f.seek(0)  # Rewind to start`

In [ ]:
with open(csv_path) as file_object:
    file_object.readline()
    print('Offset byte position:', file_object.tell())
    file_object.seek(0)
    print('Offset byte position after seek(0):', file_object.tell())


### 15. Flushing Stream Buffers (`f.flush()`)
**Explanation**: Python buffers write operations in user-space memory before flushing to the OS kernel. Calling `f.flush()` forces the internal buffer to write immediately to the operating system without closing the file handle (use `os.fsync(f.fileno())` to force OS write to physical disk).

**Syntax**: `f.flush()  # Force write buffer to OS`

In [ ]:
with open('t_flush.txt', 'w') as file_object:
    file_object.write('Buffered')
    file_object.flush()
import os
if os.path.exists('t_flush.txt'): os.remove('t_flush.txt')


## Section 3: Fintech Senior Interview Scenarios
**Explanation**: Atomic log audit suppressors, safe database transaction context managers, and memory-mapped file reading.


In [ ]:
# Solution:
with open(csv_path, 'r') as f:
    start = f.tell()
    f.readline()  # header
    row1 = f.readline()
    end = f.tell()
    print(f'First Row byte offset size: {end - start} bytes')


### Q2: Custom Exception-Suppressing Context Manager
**Explanation**: **Scenario**: Implement a custom context manager `LogAuditSuppressor` that catches and logs `IndexError` during messy record parsing while suppressing the error to keep the pipeline running.

**Syntax**: `class LogAuditSuppressor: def __exit__(self, exc_type, exc_val, tb): return exc_type is IndexError`

In [ ]:
# Solution:
class LogAuditSuppressor:
    def __enter__(self): return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        return exc_type is IndexError

with LogAuditSuppressor():
    l = []
    item = l[0]
print('IndexError suppressed successfully!')
